# DeepSHAP attributions of the fine-tuned AlphaGenome model at SDR amplicons

Analyses DeepSHAP attributions of the fine-tuned AlphaGenome `multiome_ATAC` head for the
SDR tiling amplicons 39–42 (16 kb windows):

1. Cell-type-specific attributions: each cell type's (CM, FB, ED) attributions minus the
   mean of the other two, normalised by their local activity.
2. Seqlets (TF-MoDISco) annotated with HOCOMOCO v14 motifs.
3. Attribution logos around the amplicons and the SDR variants, next to their measured logFC.

**Inputs:** the attribution arrays in `attribution_dir` (`test_amp_attributions.npy` = CM,
`test_amp_FB_attributions.npy`, `test_amp_ED_attributions.npy`; computed with
`scripts/attribution_analysis/test_AG_ft_attributions.py`, using
`cardiomics_s2f.finetuning.alphagenome.attributions.compute_deepshap_attributions`),
`genome_fasta`, `motif_meme_file`, and from `variant_data_dir` the SDR variant table and
the ENCODE4 cCREs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyranges as pr
import torch
from alphagenome.data import genome
from alphagenome.models import dna_client
from pyfaidx import Fasta
from tangermeme.annotate import annotate_seqlets
from tangermeme.io import read_meme
from tangermeme.plot import plot_logo
from tangermeme.seqlet import tfmodisco_seqlets

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.finetuning.alphagenome.attributions import (
    normalize_attributions_by_local_activity,
    one_hot_encode_sequences,
)
from cardiomics_s2f.io.variant_annotation import load_ccre_data

In [ ]:
paths = load_paths()
hg38 = Fasta(str(paths.genome_fasta))

WINDOW_SIZE = dna_client.SEQUENCE_LENGTH_16KB

## Amplicon sequences

In [ ]:
amplicons = {
    "amp39": [genome.Interval('chr5', 65051165, 65051403),
              genome.Interval('chr5', 65051165, 65051403).resize(WINDOW_SIZE)], #resizing amplicon to accepted sequnce input length
    "amp40": [genome.Interval('chr12', 51871162, 51871409),
              genome.Interval('chr12', 51871162, 51871409).resize(WINDOW_SIZE)],
    "amp41": [genome.Interval('chr17', 12121654, 12121853),
              genome.Interval('chr17', 12121654, 12121853).resize(WINDOW_SIZE)],
    "amp42": [genome.Interval('chrX', 40475093, 40475322),
              genome.Interval('chrX', 40475093, 40475322).resize(WINDOW_SIZE)],
}

amps = []
seqs = []

for amp, ivs in amplicons.items():
    amps.append(amp)
    iv = ivs[1]   
    amp_seq = str(hg38[iv.chromosome][iv.start:iv.end])
    seqs.append(amp_seq)

In [ ]:
seq_1hot = one_hot_encode_sequences(seqs) # Output: (batch, WINDOW_SIZE, 4)

## Attributions (CM, FB, ED)

In [ ]:
attributions_CM = np.load(
    paths.attribution_dir / "test_amp_attributions.npy"
)
attributions_CM.shape

In [ ]:
attributions_FB = np.load(
    paths.attribution_dir / "test_amp_FB_attributions.npy"
)
attributions_FB.shape

In [ ]:
attributions_ED = np.load(
    paths.attribution_dir / "test_amp_ED_attributions.npy"
)
attributions_ED.shape

In [ ]:
att_x_input_CM = torch.tensor(attributions_CM * seq_1hot).permute(0, 2, 1)
att_x_input_FB = torch.tensor(attributions_FB * seq_1hot).permute(0, 2, 1)
att_x_input_ED = torch.tensor(attributions_ED * seq_1hot).permute(0, 2, 1)

In [ ]:
att_x_input_CM

In [ ]:
scores = {
    "CM": att_x_input_CM,
    "FB": att_x_input_FB,
    "ED": att_x_input_ED,
}

## Cell-type-specific attributions

In [ ]:
#subtract mean scores to get celltype sceifc scores 
ct_scores = {
    k: v - torch.stack([scores[kk] for kk in scores if kk != k]).mean(dim=0)
    for k, v in scores.items()
}

In [ ]:
{k: v.shape for k, v in ct_scores.items()}

In [ ]:
ct_scores["CM"][0, ...]

In [ ]:
fig, axis = plt.subplots( len(ct_scores), 1,  figsize = (20, 15))

amp_n =  0

for n, (k,v) in enumerate(ct_scores.items()):
    start = WINDOW_SIZE//2 - 250
    end = WINDOW_SIZE//2 + 250
    plot_logo(ct_scores[k][amp_n, ...], ax=axis[n], start=start, end=end, score_key='attribution')
plt.show()

In [ ]:
#apply gaussian smoothening 
norm_ct_scores = {k: normalize_attributions_by_local_activity(v) for k, v in ct_scores.items()}

In [ ]:
fig, axis = plt.subplots( len(norm_ct_scores), 1,  figsize = (20, 15))

amp_n =  0

for n, (k,v) in enumerate(norm_ct_scores.items()):
    start = WINDOW_SIZE//2 - 250
    end = WINDOW_SIZE//2 + 250
    plot_logo(norm_ct_scores[k][amp_n, ...], ax=axis[n], start=start, end=end, score_key='attribution')
plt.show()

## Seqlets and motif annotation

In [ ]:
seqlets = {k: tfmodisco_seqlets(v.sum(dim=1), window_size=21) for k, v in norm_ct_scores.items()}

In [ ]:
meme = read_meme(str(paths.motif_meme_file))
meme_names = list(meme.keys())
meme_names[:5]

In [ ]:
annot = {k: annotate_seqlets(torch.tensor(seq_1hot).permute(0, 2, 1), v, meme, n_nearest=3) for k, v in seqlets.items()}

In [ ]:
annot_seqlets = {}
for k, (m_idx, m_p) in annot.items():
    seqlets_k = seqlets[k]
    seqlets_k["seq_idx"] = seqlets_k["example_idx"]
    seqlets_k["example_idx"] = [str([meme_names[i] for i in idx]) for idx in m_idx.tolist()]#[ meme_names[idx] for idx in motif_idxs_FB]
    seqlets_k["p_val"] =  [min(p).item() for p in m_p]
    annot_seqlets[k] = seqlets_k

In [ ]:
plt.plot(annot_seqlets["CM"]['attribution'].sort_values().values)
plt.xlabel("Hit (Sorted by Attribution)")
plt.ylabel("Attribution Sum")
plt.hlines(y=-150, xmin = 0, xmax=len(annot_seqlets["CM"]))
plt.hlines(y=150, xmin = 0, xmax=len(annot_seqlets["CM"]))
plt.title("Attribution Score per Hit")
plt.show()

In [ ]:
annot_seqlets = {k: v[v.attribution.abs() >= 150] for k,v in annot_seqlets.items()}

In [ ]:
# seqlets = seqlets[seqlets.attribution.abs() >= 15]
# # seqlets_FB = seqlets[seqlets_FB.attribution.abs() >= 15]
# # seqlets_ED = seqlets[seqlets_ED.attribution.abs() >= 15]

In [ ]:
annot_seqlets["CM"][(annot_seqlets["CM"].start > WINDOW_SIZE//2 - 100) & (annot_seqlets["CM"].end < WINDOW_SIZE//2 + 100)] 

In [ ]:
annot_seqlets["FB"][(annot_seqlets["FB"].start > WINDOW_SIZE//2 - 100) & (annot_seqlets["FB"].end < WINDOW_SIZE//2 + 100)] 

In [ ]:
annot_seqlets["ED"][(annot_seqlets["ED"].start > WINDOW_SIZE//2 - 100) & (annot_seqlets["ED"].end < WINDOW_SIZE//2 + 100)] 

## Attributions per amplicon

**amp39 chr5 overalps withthe satr/intron of an lncRNA ENSG00000300387 that is sense with CWC27 which leads to an psotive effect in ED!** 

In [ ]:
#cm
ct = "CM"

fig, axis = plt.subplots( len(amps), 1,  figsize = (20, 12))
for n, amp in enumerate(amps):
    amp_len = amplicons[amp][0].end - amplicons[amp][0].start
    start = WINDOW_SIZE//2 - amp_len // 2
    end = start + amp_len
    plot_logo(norm_ct_scores[ct][n], ax=axis[n], start=start, end=end, annotations=annot_seqlets[ct][annot_seqlets[ct].seq_idx == n], score_key='attribution')
plt.show()

In [ ]:
#cm
ct = "FB"

fig, axis = plt.subplots( len(amps), 1,  figsize = (20, 12))
for n, amp in enumerate(amps):
    amp_len = amplicons[amp][0].end - amplicons[amp][0].start
    start = WINDOW_SIZE//2 - amp_len // 2
    end = start + amp_len
    plot_logo(norm_ct_scores[ct][n], ax=axis[n], start=start, end=end, annotations=annot_seqlets[ct][annot_seqlets[ct].seq_idx == n], score_key='attribution')
plt.show()

In [ ]:
#cm
ct = "ED"

fig, axis = plt.subplots( len(amps), 1,  figsize = (20, 12))
for n, amp in enumerate(amps):
    amp_len = amplicons[amp][0].end - amplicons[amp][0].start
    start = WINDOW_SIZE//2 - amp_len // 2
    end = start + amp_len
    plot_logo(norm_ct_scores[ct][n], ax=axis[n], start=start, end=end, annotations=annot_seqlets[ct][annot_seqlets[ct].seq_idx == n], score_key='attribution')
plt.show()

## SDR variants in the amplicons

In [ ]:
SDR_VCF_FILE = paths.variant_data_dir / "VCF_SDR007A_SDR010ABC_Comb.csv"

sdr = pd.read_csv(SDR_VCF_FILE)
sdr = sdr[(sdr.var_type == "positive") & (sdr.cell_type.isin(["CM", "FB", "ED"])) & (sdr.sdr_experiment == "SDR010ABC")]

In [ ]:
#add metadata annoation
#cCRE
ccre_pr = load_ccre_data(paths.variant_data_dir / "hg38_ENCODE4_cCREs.gz")
ccre_pr

In [ ]:
rows = []
for amp_name, intervals in amplicons.items():
    rows.append({
            "Chromosome": intervals[0].chromosome,
            "Start": intervals[0].start,
            "End": intervals[0].end,
            "amplicon_id": amp_name,
})

amp_df = pd.DataFrame(rows)
amp_pr = pr.PyRanges(amp_df)
overlapping_matches = amp_pr.join(ccre_pr)

# Convert to a pandas DataFrame to look at the results
df_matches = overlapping_matches.df
df_matches

In [ ]:
x = pd.read_csv(SDR_VCF_FILE)
x[(x.CHROM == "chr5")].sort_values(by="var_type")

In [ ]:
sdr[sdr.index == 280].variant_id.values

In [ ]:
var = "chr12-51871347-C-CA-(for)"
CHROM = sdr[sdr.variant_id == var].iloc[0,:].CHROM
POS = sdr[sdr.variant_id == var].iloc[0,:].POS

for amp_name, intervals in amplicons.items():
    if intervals[1].chromosome == CHROM and intervals[1].start <= POS <= intervals[1].end:
        iv = intervals[1]
        amp = amp_name
        
fig, axes = plt.subplots(3, 1, figsize = (10, 6))

size = 20
var_loc = POS - iv.start
start = var_loc - size
end = var_loc + size
n = np.where(np.array(amps) == amp)[0].item()

vmax = max([v.sum(dim=1)[n, start:end].max().item() for v in norm_ct_scores.values()])
vmin = min([v.sum(dim=1)[n, start:end].min().item() for v in norm_ct_scores.values()])

for ax, (k, v) in enumerate(norm_ct_scores.items()):
    plot_logo(v[n, ...], ax=axes[ax], start=start, end=end, annotations=annot_seqlets[k][annot_seqlets[k].seq_idx == n], score_key='attribution')
    axes[ax].set_ylim(vmin, vmax)
    axes[ax].set_ylabel(f"{k} specific attribution")

plt.suptitle(f"Attribution for {amp} around {var} ({2*size}bp window)")    
plt.tight_layout()
plt.show()

In [ ]:
amplicons[amp]

In [ ]:
var = "chr12-51871347-C-CA-(for)"
x = pd.read_csv(SDR_VCF_FILE)
df = x[(x.variant_id == var) & (x.cell_type.isin(["CM", "FB", "ED"])) & (x.sdr_experiment == "SDR010ABC")]
df = df.groupby(["Gene", "cell_type", "variant_id", "var_type"], as_index=False).logFC.mean()
color_map = {"positive": "red", "negative": "grey"}
df["color"] = df["var_type"].map(color_map)
variant_title = df["variant_id"].iloc[0]
plt.figure(figsize=(4, 4))
for _, row in df.iterrows():
    plt.scatter(row["cell_type"],row["logFC"],color=row["color"], s=80,  edgecolor="black", alpha=0.8)
for _, row in df.iterrows():
    plt.text(row["cell_type"],row["logFC"], row["Gene"],fontsize=8, ha="left", va="bottom")
plt.axhline(0, linestyle="--", color="black", linewidth=1)
plt.xlabel("Cell Type")
plt.ylabel("logFC")
plt.xlim(-0.25, 1.25)
plt.title(variant_title)
plt.tight_layout()
plt.show()

In [ ]:
var = 'chr5-65051259-GCTTGCTTCA-G-(for)'

CHROM = sdr[sdr.variant_id == var].iloc[0,:].CHROM
POS = sdr[sdr.variant_id == var].iloc[0,:].POS

for amp_name, intervals in amplicons.items():
    if intervals[1].chromosome == CHROM and intervals[1].start <= POS <= intervals[1].end:
        iv = intervals[1]
        amp = amp_name
        
fig, axes = plt.subplots(3, 1, figsize = (10, 6))

size = 60
var_loc = POS - iv.start
start = var_loc - 20
end = var_loc + size
n = np.where(np.array(amps) == amp)[0].item()

vmax = max([v.sum(dim=1)[n, start:end].max().item() for v in norm_ct_scores.values()])
vmin = min([v.sum(dim=1)[n, start:end].min().item() for v in norm_ct_scores.values()])

for ax, (k, v) in enumerate(norm_ct_scores.items()):
    plot_logo(v[n, ...], ax=axes[ax], start=start, end=end, annotations=annot_seqlets[k][annot_seqlets[k].seq_idx == n], score_key='attribution')
    axes[ax].set_ylim(vmin, vmax)
    axes[ax].set_ylabel(f"{k}")

plt.suptitle(f"Attribution for {amp} around {var} ({2*size}bp window)")    
plt.tight_layout()
plt.show()

In [ ]:
var = 'chr5-65051259-GCTTGCTTCA-G-(for)'
x = pd.read_csv(SDR_VCF_FILE)
df = x[(x.variant_id == var) & (x.cell_type.isin(["CM", "FB", "ED"])) & (x.sdr_experiment == "SDR010ABC")]
df = df.groupby(["Gene", "cell_type", "variant_id", "var_type"], as_index=False).logFC.mean()
color_map = {"positive": "red", "negative": "grey"}
df["color"] = df["var_type"].map(color_map)
variant_title = df["variant_id"].iloc[0]
plt.figure(figsize=(3, 4))
for _, row in df.iterrows():
    plt.scatter(row["cell_type"],row["logFC"],color=row["color"], s=80,  edgecolor="black", alpha=0.8)
for _, row in df.iterrows():
    plt.text(row["cell_type"],row["logFC"], row["Gene"],fontsize=8, ha="left", va="bottom")
plt.axhline(0, linestyle="--", color="black", linewidth=1)
plt.xlabel("Cell Type")
plt.ylabel("logFC")
plt.xlim(-0.25, 1.25)
plt.title(variant_title)
plt.tight_layout()
plt.show()

In [ ]:
var = 'chr17-12121780-T-C-(rev)|chr17-12121782-T-C-(rev)'
CHROM = sdr[sdr.variant_id == var].iloc[0,:].CHROM
POS = sdr[sdr.variant_id == var].iloc[0,:].POS

for amp_name, intervals in amplicons.items():
    if intervals[1].chromosome == CHROM and intervals[1].start <= POS <= intervals[1].end:
        iv = intervals[1]
        amp = amp_name
        
fig, axes = plt.subplots(3, 1, figsize = (10, 6))

size = 50
var_loc = POS - iv.start
start = var_loc - size
end = var_loc + size
n = np.where(np.array(amps) == amp)[0].item()

vmax = max([v.sum(dim=1)[n, start:end].max().item() for v in norm_ct_scores.values()])
vmin = min([v.sum(dim=1)[n, start:end].min().item() for v in norm_ct_scores.values()])

for ax, (k, v) in enumerate(norm_ct_scores.items()):
    plot_logo(v[n, ...], ax=axes[ax], start=start, end=end, annotations=annot_seqlets[k][annot_seqlets[k].seq_idx == n], score_key='attribution')
    axes[ax].set_ylim(vmin, vmax)
    axes[ax].set_ylabel(f"{k} Celltype specific attribution")

plt.suptitle(f"Attribution for {amp} around {var} ({2*size}bp window)")    
plt.tight_layout()
plt.show()